# 03 · Clean and Preprocess Master Meter 1

**PRT661 Assessment 2 · Group 4 · DKASC Alice Springs**
**Owner: Abhishek Tamang · Epic: Data Processing**

Notebook 02 aggregated to hourly *before* cleaning, so sentinel values propagated into the
hourly means. This notebook corrects that ordering: it cleans at 5 minute resolution and then
rebuilds the hourly store from clean data.

Stages:

1. **Sentinel removal**, applied to instrument channels only.
2. **Cumulative register diagnosis and repair**, handled separately because a counter is not an
   instrument reading and does not obey instrument bounds.
3. **Physical range enforcement** on every measured channel.
4. **Short gap interpolation**, up to 15 minutes; longer gaps stay missing.
5. **Hourly rebuild** requiring at least 6 of 12 readings per hour.

Emits `Datasets/processed/mastermeter1_hourly_clean.parquet` and
`Outputs/cleaning_report_mastermeter1.md`.

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO = Path("/Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting")
OUT  = REPO/"Outputs"
PROC = REPO/"Datasets"/"processed"

TARGET  = "Active_Power"
COUNTER = "Active_Energy_Delivered_Received"

raw = pd.read_parquet(PROC/"mastermeter1_5min.parquet")
raw = raw.drop(columns=[c for c in ["interval_energy"] if c in raw.columns])

print(f"{len(raw):,} rows x {raw.shape[1]} cols")
print(f"{raw.index.min()} to {raw.index.max()}")

before = pd.DataFrame({
    "missing_before_%": (100*raw.isna().mean()).round(2),
    "min_before":       raw.min().round(3),
    "max_before":       raw.max().round(3),
})

1,782,422 rows x 16 cols
2008-09-12 05:55:00 to 2025-08-23 05:00:00


## 1 · Sentinel removal, instrument channels only

Loggers write fixed error codes rather than blanks: `-99999` appears in wind direction and
values in the tens of thousands in tilted radiation. Any magnitude at or above 9999 in a
*measured* channel is treated as a code, because no instrument here has a legitimate reading of
that size.

`Active_Energy_Delivered_Received` is deliberately **excluded** from this rule. It is a
cumulative kWh register, not a measurement: at a mean load of roughly 45 kW over seventeen years
it legitimately accumulates millions of kWh. Applying an instrument bound to a counter deletes
the counter. It is diagnosed separately in the next section.

In [11]:
SENTINEL_ABS = 9999.0
EXEMPT = {COUNTER}          # cumulative register, not an instrument reading

df = raw.copy()
sentinel_hits = {}
for c in df.columns:
    if c in EXEMPT:
        continue
    mask = df[c].abs() >= SENTINEL_ABS
    n = int(mask.sum())
    if n:
        sentinel_hits[c] = n
        df.loc[mask, c] = np.nan

print(f"{'column':<36} {'sentinels removed':>18}")
print("-" * 56)
for c, n in sorted(sentinel_hits.items(), key=lambda kv: -kv[1]):
    print(f"{c:<36} {n:>18,}")
print(f"{'TOTAL':<36} {sum(sentinel_hits.values()):>18,}")
print(f"\nexempt from this rule: {sorted(EXEMPT)}")

column                                sentinels removed
--------------------------------------------------------
Wind_Direction                                    1,466
Radiation_Diffuse_Tilted                              6
TOTAL                                             1,472

exempt from this rule: ['Active_Energy_Delivered_Received']


## 2 · Diagnose the cumulative register

Before repairing the counter, establish what it actually is. Three hypotheses are
distinguishable from the data:

* If `999999` appears many times at scattered dates, it is an error code.
* If the series climbs to `999999` and restarts near zero, it is a six digit register that wraps.
* Isolated negatives or wild values are corrupt reads regardless.

The diff distribution settles it. A wrap produces differences near minus one million; a corrupt
read produces arbitrary ones.

In [12]:
cnt = raw[COUNTER]

print(f"present readings   : {int(cnt.notna().sum()):,}")
print(f"range              : {cnt.min():,.3f} to {cnt.max():,.3f}")
print(f"exactly 999999.000 : {int((cnt == 999999.0).sum()):,}")
print(f"negative readings  : {int((cnt < 0).sum()):,}")
print()
print("value quantiles:")
print(cnt.quantile([0, .01, .25, .5, .75, .99, 1]).round(1).to_string())

d = cnt.diff()
print()
print(f"differences: {int(d.notna().sum()):,}")
print(f"   near -1,000,000 (wrap candidates) : {int((d < -900_000).sum()):,}")
print(f"   other negatives (corrupt reads)   : {int(((d < 0) & (d >= -900_000)).sum()):,}")
print(f"   above 1000 kWh per 5 min          : {int((d > 1000).sum()):,}")
print()
print("difference quantiles:")
print(d.quantile([0, .001, .25, .5, .75, .999, 1]).round(3).to_string())

present readings   : 1,740,736
range              : -68.166 to 999,999.000
exactly 999999.000 : 1
negative readings  : 4,079

value quantiles:
0.00       -68.2
0.01      5074.0
0.25    217014.9
0.50    449978.5
0.75    710226.8
0.99    988553.8
1.00    999999.0

differences: 1,725,449
   near -1,000,000 (wrap candidates) : 6
   other negatives (corrupt reads)   : 743,880
   above 1000 kWh per 5 min          : 0

difference quantiles:
0.000   -999987.738
0.001        -0.125
0.250        -0.043
0.500         0.000
0.750         7.406
0.999        16.625
1.000       461.312


## 3 · Repair the register and derive interval energy

Wraps have one million added back. Remaining negatives and physically impossible jumps become
missing rather than being clipped to zero, which would fabricate intervals of no generation.

`Active_Power` is also integrated directly into energy as an independent estimate. Agreement
between the two is a validation check on the repair; the power derived series is the one carried
forward, because it inherits the target's 2.34% missingness rather than the register's faults.

In [13]:
cnt_fixed = cnt.copy()
cnt_fixed[cnt_fixed < 0] = np.nan          # corrupt reads

d = cnt_fixed.diff()
n_wrap = int((d < -900_000).sum())
d = d.where(d > -900_000, d + 1_000_000)   # undo six digit wrap

n_neg  = int((d < 0).sum())
n_huge = int((d > 1000).sum())
d[(d < 0) | (d > 1000)] = np.nan

df[COUNTER] = cnt_fixed
df["interval_energy"] = d
df["energy_from_power"] = df[TARGET] * (5/60)     # kW over 5 minutes -> kWh

print(f"register wraps corrected  : {n_wrap:,}")
print(f"corrupt differences nulled: {n_neg + n_huge:,}")
print(f"interval_energy usable    : {int(df['interval_energy'].notna().sum()):,} "
      f"({100*df['interval_energy'].notna().mean():.2f}%)")
print()
tot_c = df["interval_energy"].sum()
tot_p = df["energy_from_power"].sum()
print(f"total energy from register    : {tot_c:>14,.0f} kWh")
print(f"total energy from Active_Power: {tot_p:>14,.0f} kWh")
print(f"agreement                     : {100*min(tot_c,tot_p)/max(tot_c,tot_p):>13.1f}%")

register wraps corrected  : 6
corrupt differences nulled: 740,055
interval_energy usable    : 981,315 (55.06%)

total energy from register    :      6,467,319 kWh
total energy from Active_Power:      6,488,854 kWh
agreement                     :          99.7%


## 4 · Physical range enforcement

Bounds come from instrument physics and the Alice Springs climate record, not from the data's own percentiles, because percentiles let a systematic fault define normality. Recorded temperature extremes for the town are roughly -7.5 and 45.2 °C.

In [16]:
RANGES = {
    "Active_Power":                    (-20,  400),
    "Current_Phase_Average":           (0,    500),
    "Power_Factor_Signed":             (-100, 100),
    "Average_Voltage_Line_to_Neutral": (180,  290),
    "Frequency":                       (45,   55),
    "THD_Voltage_Average":             (0,    30),
    "Wind_Speed":                      (0,    50),
    "Weather_Temperature_Celsius":     (-5,   50),
    "Weather_Relative_Humidity":       (0,    100),
    "Global_Horizontal_Radiation":     (0,    1400),
    "Diffuse_Horizontal_Radiation":    (0,    1200),
    "Wind_Direction":                  (0,    360),
    "Weather_Daily_Rainfall":          (0,    500),
    "Radiation_Global_Tilted":         (0,    1600),
    "Radiation_Diffuse_Tilted":        (0,    1200),
}

print(f"{'column':<36} {'low':>8} {'high':>8} {'nulled':>12} {'%':>9}")
print("-" * 76)
range_hits = {}
for c, (lo, hi) in RANGES.items():
    if c not in df.columns:
        continue
    s = df[c]
    mask = ((s < lo) | (s > hi)) & s.notna()
    n = int(mask.sum())
    range_hits[c] = n
    df.loc[mask, c] = np.nan
    print(f"{c:<36} {lo:>8} {hi:>8} {n:>12,} {100*n/len(df):>8.3f}%")
print(f"{'TOTAL':<36} {'':>8} {'':>8} {sum(range_hits.values()):>12,}")

column                                    low     high       nulled         %
----------------------------------------------------------------------------
Active_Power                              -20      400            0    0.000%
Current_Phase_Average                       0      500            0    0.000%
Power_Factor_Signed                      -100      100            0    0.000%
Average_Voltage_Line_to_Neutral           180      290            0    0.000%
Frequency                                  45       55            0    0.000%
THD_Voltage_Average                         0       30            0    0.000%
Wind_Speed                                  0       50            0    0.000%
Weather_Temperature_Celsius                -5       50            0    0.000%
Weather_Relative_Humidity                   0      100            0    0.000%
Global_Horizontal_Radiation                 0     1400            0    0.000%
Diffuse_Horizontal_Radiation                0     1200           

## 5 · Short gap interpolation

A 5 minute gap in irradiance is safely bridged; a six hour gap is not. Time weighted
interpolation is applied only where at most 3 consecutive slots are missing, which is 15 minutes.

The target is deliberately **not** interpolated. Filling the variable being predicted
manufactures agreement between model and data.

In [17]:
MAX_GAP = 3
SKIP = {TARGET, "interval_energy", "energy_from_power", COUNTER}

def gap_lengths(s):
    isna = s.isna()
    grp = (isna != isna.shift()).cumsum()
    return isna.groupby(grp).transform("size").where(isna, 0)

interp_hits = {}
for c in df.columns:
    if c in SKIP or df[c].notna().sum() == 0:
        continue
    s = df[c]
    fillable = s.isna() & (gap_lengths(s) <= MAX_GAP)
    n = int(fillable.sum())
    if n:
        filled = s.interpolate(method="time", limit=MAX_GAP, limit_area="inside")
        df.loc[fillable, c] = filled[fillable]
        interp_hits[c] = n

print(f"{'column':<36} {'values interpolated':>20}")
print("-" * 58)
for c, n in sorted(interp_hits.items(), key=lambda kv: -kv[1]):
    print(f"{c:<36} {n:>20,}")
print(f"{'TOTAL':<36} {sum(interp_hits.values()):>20,}")
print(f"\n{TARGET} not interpolated: {int(df[TARGET].isna().sum()):,} still missing")

column                                values interpolated
----------------------------------------------------------
Wind_Direction                                     25,742
Radiation_Diffuse_Tilted                           21,977
Radiation_Global_Tilted                            21,968
Frequency                                          21,522
Average_Voltage_Line_to_Neutral                    21,519
Power_Factor_Signed                                21,465
Current_Phase_Average                              21,151
THD_Voltage_Average                                21,151
Global_Horizontal_Radiation                        20,895
Weather_Temperature_Celsius                        20,878
Diffuse_Horizontal_Radiation                       20,836
Weather_Daily_Rainfall                             20,822
Weather_Relative_Humidity                          20,650
Wind_Speed                                            779
TOTAL                                             281,355

Active_Power

## 6 · Rebuild hourly from clean data

An hourly mean over two surviving readings is not comparable to one over twelve. Hours below 6 of 12 readings are nulled, and `n_obs` is retained so the modelling notebook can filter further.

In [18]:
MIN_OBS = 6

WEATHER = ["Wind_Speed", "Weather_Temperature_Celsius", "Weather_Relative_Humidity",
           "Global_Horizontal_Radiation", "Diffuse_Horizontal_Radiation", "Wind_Direction",
           "Weather_Daily_Rainfall", "Radiation_Global_Tilted", "Radiation_Diffuse_Tilted"]
ELECTRICAL = ["Active_Power", "Current_Phase_Average", "Power_Factor_Signed",
              "Average_Voltage_Line_to_Neutral", "Frequency", "THD_Voltage_Average"]

agg = {c: "mean" for c in WEATHER + ELECTRICAL if c in df.columns}
agg["interval_energy"]   = "sum"
agg["energy_from_power"] = "sum"

hourly = df.resample("1h").agg(agg)
hourly["n_obs"] = df[TARGET].resample("1h").count()
hourly.loc[hourly["n_obs"] < MIN_OBS, [c for c in hourly.columns if c != "n_obs"]] = np.nan

print(f"hourly rows                  : {len(hourly):,}")
print(f"hours with >= {MIN_OBS} readings     : {int((hourly['n_obs'] >= MIN_OBS).sum()):,}")
print(f"hours nulled (too few reads) : {int((hourly['n_obs'] < MIN_OBS).sum()):,}")
print(f"usable target hours          : {int(hourly[TARGET].notna().sum()):,}")
print()
cols = [TARGET, "energy_from_power", "Global_Horizontal_Radiation",
        "Weather_Temperature_Celsius", "n_obs"]
print(hourly[cols].describe().round(3).to_string())

hourly rows                  : 148,537
hours with >= 6 readings     : 147,134
hours nulled (too few reads) : 1,403
usable target hours          : 147,134

       Active_Power  energy_from_power  Global_Horizontal_Radiation  Weather_Temperature_Celsius       n_obs
count    147134.000         147134.000                   146109.000                   145859.000  148537.000
mean         44.756             44.079                      261.966                       21.273      11.719
std          61.320             60.570                      354.913                        9.557       1.315
min          -1.787             -1.787                        0.000                       -4.865       0.000
25%          -0.476             -0.476                        2.852                       14.553      12.000
50%           0.096              0.094                       11.806                       21.723      12.000
75%          89.548             87.519                      527.017               

## 7 · Before and after

In [19]:
after = pd.DataFrame({
    "missing_after_%": (100*df.isna().mean()).round(2),
    "min_after":       df.min().round(3),
    "max_after":       df.max().round(3),
})
comparison = before.join(after, how="outer")
comparison["delta_missing_pp"] = (comparison["missing_after_%"]
                                  - comparison["missing_before_%"]).round(2)
comparison = comparison[["missing_before_%", "missing_after_%", "delta_missing_pp",
                         "min_before", "min_after", "max_before", "max_after"]]
pd.set_option("display.width", 220, "display.max_colwidth", 40)
print(comparison.to_string())
print("\nnegative delta = missingness fell (gaps interpolated)")
print("positive delta = values removed as invalid")

                                  missing_before_%  missing_after_%  delta_missing_pp  min_before  min_after  max_before   max_after
Active_Energy_Delivered_Received              2.34             2.57              0.23     -68.166      0.105  999999.000  999999.000
Active_Power                                  2.34             2.34              0.00      -1.861     -1.861     241.026     241.026
Average_Voltage_Line_to_Neutral               2.31             1.17             -1.14       0.000    182.379     269.254     269.254
Current_Phase_Average                        41.67            40.48             -1.19       0.000      0.000     366.075     366.075
Diffuse_Horizontal_Radiation                  2.96             1.79             -1.17       0.000      0.000    2134.985     867.347
Frequency                                     2.35             1.17             -1.18       0.000     47.636      53.000      53.000
Global_Horizontal_Radiation                   2.96             1.79  

## 8 · Write the cleaned store and the cleaning report

In [20]:
p5c = PROC/"mastermeter1_5min_clean.parquet"
p1c = PROC/"mastermeter1_hourly_clean.parquet"
df.to_parquet(p5c, compression="snappy")
hourly.to_parquet(p1c, compression="snappy")
print(f"{p5c.name:<38} {p5c.stat().st_size/1e6:>8.1f} MB   ({len(df):,} rows)")
print(f"{p1c.name:<38} {p1c.stat().st_size/1e6:>8.1f} MB   ({len(hourly):,} rows)")

n_sent, n_rng, n_int = (sum(sentinel_hits.values()), sum(range_hits.values()),
                        sum(interp_hits.values()))

lines = [
    "# Cleaning Report: Master Meter 1", "",
    "Generated by `notebooks/03_clean_mastermeter1.ipynb`.",
    "Input: `Datasets/processed/mastermeter1_5min.parquet`.", "",
    "## What was applied", "",
    "| Stage | Rule | Values affected |", "|---|---|---|",
    f"| Sentinel removal | instrument channels, absolute value at or above {SENTINEL_ABS:,.0f} | {n_sent:,} |",
    f"| Register wrap correction | six digit rollover, one million added back | {n_wrap:,} |",
    f"| Register corrupt reads | negatives and jumps above 1000 kWh per 5 min nulled | {n_neg + n_huge:,} |",
    f"| Physical range | per column domain bounds | {n_rng:,} |",
    f"| Short gap interpolation | time weighted, gaps up to {MAX_GAP} slots (15 min) | {n_int:,} |",
    f"| Hourly threshold | hour nulled below {MIN_OBS} of 12 readings | {int((hourly['n_obs'] < MIN_OBS).sum()):,} hours |",
    "",
    f"The target `{TARGET}` was not interpolated at any stage.",
    "The cumulative register was excluded from the sentinel rule: it is a counter, not an",
    "instrument reading, and legitimately exceeds any instrument bound.", "",
    "## Applied ranges", "", "| Column | Lower | Upper |", "|---|---|---|",
]
lines += [f"| `{c}` | {lo} | {hi} |" for c, (lo, hi) in RANGES.items() if c in df.columns]
lines += ["", "## Before and after, per column", "",
          "| Column | Missing before % | Missing after % | Delta pp | Min before | Min after | Max before | Max after |",
          "|---|---|---|---|---|---|---|---|"]
for c, r in comparison.iterrows():
    lines.append(f"| `{c}` | {r['missing_before_%']} | {r['missing_after_%']} | "
                 f"{r['delta_missing_pp']} | {r['min_before']} | {r['min_after']} | "
                 f"{r['max_before']} | {r['max_after']} |")
lines += ["", "## Resulting hourly store", "", "| Measure | Value |", "|---|---|",
          f"| Hourly rows | {len(hourly):,} |",
          f"| Usable target hours | {int(hourly[TARGET].notna().sum()):,} |",
          f"| Coverage | {hourly.index.min()} to {hourly.index.max()} |"]

(OUT/"cleaning_report_mastermeter1.md").write_text("\n".join(lines))
print(f"\nwrote {OUT/'cleaning_report_mastermeter1.md'}")

mastermeter1_5min_clean.parquet           164.6 MB   (1,782,422 rows)
mastermeter1_hourly_clean.parquet          18.5 MB   (148,537 rows)

wrote /Users/uttamshrestha/Desktop/Data Science Practice Unit/PRT661-solar-forecasting/Outputs/cleaning_report_mastermeter1.md
